# Parcial 1 - Clasificacion de Tipos de Flores

**Objetivo:** construir un modelo de vision artificial (CNN) para clasificar imagenes del dataset `flower_photos` y dejar listos los artefactos para despliegue en Streamlit.

In [2]:
# Librerias base para manejo de datos, visualizacion y deep learning
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import classification_report, confusion_matrix

# Semillas para resultados mas reproducibles
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow:', tf.__version__)

TensorFlow: 2.19.0


En esta celda se importan las herramientas necesarias y se fija una semilla para mejorar la reproducibilidad del entrenamiento. Si no hay errores, el entorno esta listo para continuar.

## Carga del Dataset

In [3]:
# Cargar dataset desde Google Drive (Colab)
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Ruta donde moviste la carpeta flower_photos
data_dir = Path('/content/drive/MyDrive/Ciencia de datos/flower_photos')
assert data_dir.exists(), f'No se encontro la carpeta: {data_dir}'

# Parametros del pipeline de entrada
img_height, img_width = 180, 180
batch_size = 32
val_split = 0.2

# Crear datasets de entrenamiento y validacion desde carpetas
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=val_split,
    subset='training',
    seed=SEED,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=val_split,
    subset='validation',
    seed=SEED,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

# Las etiquetas se toman de los nombres de carpetas dentro de flower_photos
class_names = train_ds.class_names
num_classes = len(class_names)

print('Clases detectadas:', class_names)
print('Numero de clases:', num_classes)

Mounted at /content/drive


Mounted at /content/drive


AssertionError: No se encontro la carpeta: /content/drive/MyDrive/Ciencia de datos/flower_photos

Aqui se crean los conjuntos de entrenamiento y validacion leyendo directamente desde Google Drive en `MyDrive/Ciencia de datos/flower_photos`. Cada subcarpeta representa una clase y su nombre se usa como etiqueta (`class_names`).

## Exploracion Visual del Dataset

In [ ]:
# Visualizar un lote de imagenes para inspeccion rapida de calidad y etiquetas
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.tight_layout()
plt.show()

Esta inspeccion permite validar que las imagenes se cargan correctamente y que la etiqueta coincide con el contenido visual. Es un control antes del entrenamiento para detectar errores de organizacion en carpetas o ruido excesivo.

## Preprocesamiento y Rendimiento del Pipeline

In [ ]:
# Capa para normalizar pixeles desde [0, 255] hacia [0, 1]
normalization_layer = layers.Rescaling(1.0 / 255)

# Ajustes de rendimiento: cache y prefetch para acelerar entrenamiento
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# Verificacion rapida del rango de entrada despues de normalizar
for batch_images, _ in train_ds.take(1):
    print('Min pixel:', float(tf.reduce_min(batch_images)))
    print('Max pixel:', float(tf.reduce_max(batch_images)))

La normalizacion estabiliza el aprendizaje y evita gradientes desbalanceados. El uso de `cache` y `prefetch` reduce esperas por lectura de disco y mejora el tiempo por epoca.

## Construccion del Modelo CNN

In [ ]:
# Data augmentation para mejorar generalizacion del modelo
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

# Arquitectura CNN para clasificacion multiclase
model = keras.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),
    data_augmentation,

    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()

Se utiliza una CNN con bloques Conv2D + MaxPooling para extraer patrones morfologicos (petalos, textura, forma). Las capas `Dropout` ayudan a reducir sobreajuste y la salida `softmax` permite obtener distribucion de probabilidades por clase.

## Compilacion y Entrenamiento

In [ ]:
# Compilar con Adam y perdida de clasificacion multiclase con etiquetas enteras
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks para detener temprano y guardar el mejor modelo
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint('best_flower_model.keras', monitor='val_accuracy', save_best_only=True)
]

EPOCHS = 20
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

En esta etapa el modelo aprende a separar clases de flores. Si `val_accuracy` sube y `val_loss` baja de forma estable, el comportamiento es sano. Si hay gran brecha entre entrenamiento y validacion, podria existir sobreajuste.

## Evaluacion Cuantitativa

In [ ]:
# Metricas globales en validacion
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f'Val loss: {val_loss:.4f}')
print(f'Val accuracy: {val_accuracy:.4f}')

# Curvas de aprendizaje para analizar estabilidad del entrenamiento
hist = history.history
epochs_range = range(1, len(hist['loss']) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, hist['loss'], label='Train Loss')
plt.plot(epochs_range, hist['val_loss'], label='Val Loss')
plt.title('Loss por epoca')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, hist['accuracy'], label='Train Acc')
plt.plot(epochs_range, hist['val_accuracy'], label='Val Acc')
plt.title('Accuracy por epoca')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

El valor de accuracy resume el desempeno general, mientras que las curvas permiten ver la dinamica del entrenamiento. Un comportamiento deseable es convergencia sin oscilaciones fuertes y sin separacion extrema entre train/val.

## Matriz de Confusion y Reporte por Clase

In [ ]:
# Recolectar etiquetas reales y predichas para analisis por clase
y_true = np.concatenate([y.numpy() for _, y in val_ds], axis=0)
y_pred_prob = model.predict(val_ds, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)

# Matriz de confusion
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('Matriz de confusion - Validacion')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Reporte detallado por clase
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

Esta salida permite identificar clases con mayor confusion. El reporte de precision, recall y F1 por clase facilita justificar decisiones de mejora (mas datos por clase, augmentation adicional o ajuste de arquitectura).

## Guardado de Artefactos para Despliegue

In [ ]:
# Crear carpeta de salida para artefactos del modelo
artifacts_dir = Path('artefactos_parcial1')
artifacts_dir.mkdir(parents=True, exist_ok=True)

# Guardar modelo final
model_path = artifacts_dir / 'flower_classifier.keras'
model.save(model_path)

# Guardar clases en JSON para consumirlas facilmente desde Streamlit
classes_path = artifacts_dir / 'class_names.json'
with open(classes_path, 'w', encoding='utf-8') as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

print('Modelo guardado en:', model_path.resolve())
print('Clases guardadas en:', classes_path.resolve())

Con estos archivos (`flower_classifier.keras` y `class_names.json`) queda preparada la base para una app interactiva en Streamlit: carga de imagen, listado de clases y visualizacion de probabilidades por clase.

## Prueba de Inferencia con Distribucion de Probabilidades

In [ ]:
# Tomar una imagen del set de validacion para simular carga de usuario en app
for batch_images, _ in val_ds.take(1):
    sample_image = batch_images[0].numpy()
    break

# El modelo retorna la distribucion completa de probabilidades
probs = model.predict(np.expand_dims(sample_image, axis=0), verbose=0)[0]
pred_idx = int(np.argmax(probs))
pred_class = class_names[pred_idx]

# Visualizar imagen y clase mas probable
plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f'Prediccion: {pred_class}')
plt.axis('off')
plt.show()

# Grafico de probabilidades por clase con resaltado del maximo
colors = ['#90caf9'] * len(class_names)
colors[pred_idx] = '#ff8a65'

plt.figure(figsize=(10, 4))
bars = plt.bar(class_names, probs, color=colors)
plt.title('Distribucion de probabilidades por clase')
plt.ylabel('Probabilidad')
plt.ylim(0, 1)
plt.xticks(rotation=45, ha='right')

for bar, p in zip(bars, probs):
    plt.text(bar.get_x() + bar.get_width() / 2, p + 0.01, f'{p:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

print('Clase mas probable:', pred_class)

Este bloque replica exactamente la logica requerida para la app interactiva: una imagen de entrada, catalogo de clases, distribucion de probabilidades y resaltado automatico de la clase ganadora.

## Conclusiones de la Primera Parte

Se desarrollo el cuaderno de extremo a extremo para clasificacion de flores con CNN, incluyendo preparacion de datos, entrenamiento, evaluacion y generacion de artefactos para despliegue. Con esto queda lista la base para la segunda parte: implementacion de la interfaz en Streamlit.